In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import LeaveOneGroupOut
from captum.attr import IntegratedGradients, FeatureAblation
import matplotlib.pyplot as plt
import seaborn as sns

data = np.load('preprocessed_data.npz')
X_cnn_input = data['X_cnn'] # Shape: (batch, 32, 7680)
y_flat_bin = data['y_bin']
X_cnn_input = np.nan_to_num(X_cnn_input, nan=0.0) 

num_subjects = X_cnn_input.shape[0] // 40
groups = np.repeat(np.arange(num_subjects), 40)
logo = LeaveOneGroupOut()

high_val = (y_flat_bin == 1).sum()
low_val = (y_flat_bin == 0).sum()
pos_weight_val = torch.tensor([low_val / high_val], dtype=torch.float32).cuda()
print(f"Loaded Data. Using Class Weight: {pos_weight_val.item():.4f}")

RuntimeError: Only a single TORCH_LIBRARY can be used to register the namespace prims; please put all of your definitions in a single TORCH_LIBRARY block.  If you were trying to specify implementations, consider using TORCH_LIBRARY_IMPL (which can be duplicated).  If you really intended to define operators for a single namespace in a distributed way, you can use TORCH_LIBRARY_FRAGMENT to explicitly indicate this.  Previous registration of TORCH_LIBRARY was registered at /dev/null:241; latest registration was registered at /dev/null:241

In [ ]:


# ==========================================
# 2. ERTNet MODEL DEFINITION
# ==========================================
class ERTNet(nn.Module):
    def __init__(self, in_channels=32):
        super(ERTNet, self).__init__()
        # Module 1: Temporal Convolution for feature extraction
        self.temp_conv = nn.Conv1d(in_channels, 64, kernel_size=64, padding=31)
        self.bn1 = nn.BatchNorm1d(64)
        
        # Module 2: Spatial mapping (Channel fusion)
        self.spatial_conv = nn.Conv1d(64, 64, kernel_size=1)
        self.bn2 = nn.BatchNorm1d(64)
        
        # Pooling to reduce sequence length before the Transformer
        self.pool = nn.MaxPool1d(16) 
        
        # Module 3: Multi-Head Self-Attention (Transformer mechanism)
        self.attention = nn.MultiheadAttention(embed_dim=64, num_heads=4, batch_first=True)
        
        # Classifier (No Sigmoid here because of BCEWithLogitsLoss)
        self.fc = nn.Linear(64, 1)

    def forward(self, x, return_attention=False):
        # x: (batch, 32, 7680)
        x = torch.relu(self.bn1(self.temp_conv(x)))
        x = torch.relu(self.bn2(self.spatial_conv(x)))
        x = self.pool(x)
        
        # Prepare for attention: (batch, seq_len, features)
        x = x.permute(0, 2, 1) 
        
        attn_out, attn_weights = self.attention(x, x, x)
        
        # Global Average Pooling
        out = attn_out.mean(dim=1) 
        logits = self.fc(out)
        
        if return_attention:
            return logits, attn_weights
        return logits

# ==========================================
# 3. TRAINING LOOP (LOGO)
# ==========================================
print("Starting Leave-One-Group-Out Training...")
ertnet_results = []

for train_idx, test_idx in logo.split(X_cnn_input, y_flat_bin, groups):
    subject_id = groups[test_idx][0] + 1
    
    X_train_cpu = torch.tensor(X_cnn_input[train_idx], dtype=torch.float32)
    y_train_cpu = torch.tensor(y_flat_bin[train_idx], dtype=torch.float32).unsqueeze(1)
    train_loader = DataLoader(TensorDataset(X_train_cpu, y_train_cpu), batch_size=32, shuffle=True)
    
    model_ert = ERTNet().cuda()
    optimizer = optim.Adam(model_ert.parameters(), lr=0.0005)
    
    # Using the weighted loss function!
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_val)
    
    model_ert.train()
    for epoch in range(15):
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.cuda(), batch_y.cuda()
            
            optimizer.zero_grad()
            outputs = model_ert(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
    # Evaluation
    model_ert.eval()
    with torch.no_grad():
        X_test = torch.tensor(X_cnn_input[test_idx], dtype=torch.float32).cuda()
        y_test = torch.tensor(y_flat_bin[test_idx], dtype=torch.float32).unsqueeze(1).cuda()
        
        # Apply sigmoid manually for accuracy calculation
        raw_outputs = model_ert(X_test)
        preds = (torch.sigmoid(raw_outputs) > 0.5).float()
        acc = (preds == y_test).float().mean().item()
        
    print(f"Subject {subject_id} Accuracy: {acc*100:.2f}%")
    ertnet_results.append({'Subject': subject_id, 'Accuracy': acc})

avg_acc = np.mean([r['Accuracy'] for r in ertnet_results])
print(f"\nERTNet Subject-Independent Mean Accuracy: {avg_acc*100:.2f}%")

# ==========================================
# 4. EXPLAINABILITY (4 Techniques)
# ==========================================
print("\nGenerating XAI Visualizations...")
deap_channels = ['Fp1', 'AF3', 'F3', 'F7', 'FC5', 'FC1', 'C3', 'T7', 'CP5', 'CP1', 'P3', 'P7', 'PO3', 'O1', 'Oz', 'Pz', 'Fp2', 'AF4', 'Fz', 'F4', 'F8', 'FC6', 'FC2', 'C4', 'T8', 'CP6', 'CP2', 'P4', 'P8', 'PO4', 'O2', 'Cz']

sample_input = torch.tensor(X_cnn_input[:5], dtype=torch.float32).cuda()
sample_input.requires_grad_()

# --- XAI 1: Attention Map Visualization ---
model_ert.eval()
_, attn_weights = model_ert(sample_input, return_attention=True)
avg_attn = attn_weights.mean(dim=0).cpu().detach().numpy()

plt.figure(figsize=(10, 6))
plt.plot(avg_attn.mean(axis=1), color='teal', linewidth=2)
plt.fill_between(range(len(avg_attn)), avg_attn.mean(axis=1), alpha=0.3, color='teal')
plt.title("XAI 1: ERTNet Temporal Attention Focus")
plt.xlabel("Sequence Time Step (Post-Pooling)")
plt.ylabel("Attention Weight")
plt.show()

# --- XAI 2: Native PyTorch Saliency ---
model_ert.zero_grad()
outputs = model_ert(sample_input)
outputs.backward(torch.ones_like(outputs))
saliency_attr = sample_input.grad.abs().mean(dim=(0, 2)).cpu().detach().numpy()
saliency_attr = saliency_attr / saliency_attr.max()

plt.figure(figsize=(12, 5))
plt.bar(deap_channels[:32], saliency_attr, color='darkorange')
plt.title("XAI 2: Saliency Map - EEG Channel Importance")
plt.xticks(rotation=45)
plt.show()

# Wrapper class to hide the 'return_attention' parameter from Captum
class CaptumWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, x):
        return self.model(x, return_attention=False)

wrapped_model = CaptumWrapper(model_ert).cuda()

# Temporarily disable cuDNN to prevent LSTM/Transformer backward hooks crashing
torch.backends.cudnn.enabled = False 

# --- XAI 3: Integrated Gradients ---
ig = IntegratedGradients(wrapped_model)
ig_attr = ig.attribute(sample_input, target=0)
ig_matrix = ig_attr.abs().mean(dim=(0, 2)).cpu().detach().numpy()
ig_matrix = ig_matrix / ig_matrix.max()

# --- XAI 4: Feature Ablation ---
ablator = FeatureAblation(wrapped_model)
ablation_attr = ablator.attribute(sample_input, target=0)
ablation_matrix = ablation_attr.abs().mean(dim=(0, 2)).cpu().detach().numpy()
ablation_matrix = ablation_matrix / ablation_matrix.max()

torch.backends.cudnn.enabled = True 

# Plotting IG and Ablation side-by-side
fig, axes = plt.subplots(1, 2, figsize=(18, 5))
axes[0].bar(deap_channels[:32], ig_matrix, color='mediumseagreen')
axes[0].set_title("XAI 3: Integrated Gradients Feature Importance")
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(deap_channels[:32], ablation_matrix, color='crimson')
axes[1].set_title("XAI 4: Feature Ablation (Masking) Importance")
axes[1].tick_params(axis='x', rotation=45)
plt.show()